In [ ]:
from google.colab import files
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
import io

# =========================================================
# 1. CARGA DE ARCHIVOS (.csv de cada ciudad)
# =========================================================
print("Selecciona los archivos .csv de tus ciudades (TAMPICO, HOUSTON, etc.)")
uploaded = files.upload()

# --- FUNCIONES DE MÉTRICAS Q1 ---
def calc_smape(y_true, y_pred):
    return 100 * np.mean(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred)))

def calc_da(y_true, y_pred):
    return 100 * np.mean(np.sign(np.diff(y_true)) == np.sign(np.diff(y_pred)))

def calc_mase(y_true, y_pred, y_train_history):
    d = np.abs(np.diff(y_train_history)).sum() / (len(y_train_history) - 1)
    return mean_absolute_error(y_true, y_pred) / d

def create_windows(series, window_size=12):
    X, y = [], []
    for i in range(len(series) - window_size):
        X.append(series[i:i + window_size])
        y.append(series[i + window_size])
    return np.array(X), np.array(y)

# =========================================================
# 2. PROCESAMIENTO AUTOMÁTICO POR CIUDAD
# =========================================================
resultados_finales = []

for fn in uploaded.keys():
    ciudad_nombre = fn.replace('.csv', '').upper()
    print(f"\n>>> Procesando {ciudad_nombre}...")

    # Leer archivo evitando errores de strings/fechas
    df = pd.read_csv(io.BytesIO(uploaded[fn]))
    # Tomamos la segunda columna (temperatura) y forzamos a numérico
    series = pd.to_numeric(df.iloc[:, 1], errors='coerce').dropna().values

    # PARTICIÓN PROTOCOLO (227/65/32)
    train = series[0:227]
    val   = series[227:292]
    test  = series[292:324]
    history = np.concatenate([train, val])

    window = 12
    X_train, y_train = create_windows(train, window)
    X_val, y_val = create_windows(np.concatenate([train[-window:], val]), window)
    X_history, y_history = create_windows(history, window)
    X_test, y_test = create_windows(np.concatenate([history[-window:], test]), window)

    # --- 3. SINTONIZACIÓN ESPECIAL (GRID SEARCH) POR CIUDAD ---
    best_rmse = float('inf')
    best_params = {}

    for n_est in [100, 500]:
        for leaves in [7, 15, 31]:
            for lr in [0.01, 0.05]:
                model = lgb.LGBMRegressor(
                    n_estimators=n_est,
                    num_leaves=leaves,
                    learning_rate=lr,
                    random_state=42,
                    verbosity=-1
                ).fit(X_train, y_train)

                rmse_v = np.sqrt(mean_squared_error(y_val, model.predict(X_val)))
                if rmse_v < best_rmse:
                    best_rmse = rmse_v
                    best_params = {'n_estimators': n_est, 'num_leaves': leaves, 'learning_rate': lr}

    # --- 4. EVALUACIÓN FINAL EN TEST ---
    final_model = lgb.LGBMRegressor(**best_params, random_state=42, verbosity=-1).fit(X_history, y_history)
    preds = final_model.predict(X_test)

    # Métricas
    rmse = np.sqrt(mean_squared_error(test, preds))
    smape = calc_smape(test, preds)
    mase = calc_mase(test, preds, history)
    da = calc_da(test, preds)

    resultados_finales.append({
        "City": ciudad_nombre, "Model": "LGBM",
        "RMSE": rmse, "sMAPE": smape, "MASE": mase, "DA": da
    })

    # Exportar resultados para tu tabla maestra
    pd.DataFrame({'Actual': test, 'Forecast': preds}).to_csv(f'{ciudad_nombre}_AVG_LGBM_detailed.csv', index=False)
    print(f"  [OK] Finalizado. Mejor RMSE Val: {best_rmse:.4f}")

# --- 5. TABLA FINAL ---
df_lgbm = pd.DataFrame(resultados_finales)
print("\n=== MÉTRICAS LIGHTGBM CONSOLIDADAS ===")
print(df_lgbm.to_string(index=False))

Selecciona los archivos .csv de tus ciudades (TAMPICO, HOUSTON, etc.)


Saving TAMPICO.csv to TAMPICO.csv
Saving HOUSTON.csv to HOUSTON.csv
Saving MATAMOROS.csv to MATAMOROS.csv
Saving MIAMI.csv to MIAMI.csv

>>> Procesando TAMPICO...


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/v

  [OK] Finalizado. Mejor RMSE Val: 1.1484

>>> Procesando HOUSTON...


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/v

...

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/v

  [OK] Finalizado. Mejor RMSE Val: 1.5040

>>> Procesando MIAMI...


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


  [OK] Finalizado. Mejor RMSE Val: 1.3213

=== MÉTRICAS LIGHTGBM CONSOLIDADAS ===
     City Model     RMSE    sMAPE     MASE        DA
  TAMPICO  LGBM 1.303141 4.317532 0.579632 64.516129
  HOUSTON  LGBM 1.929629 7.126736 0.468211 87.096774
MATAMOROS  LGBM 1.849882 6.348923 0.637474 80.645161
    MIAMI  LGBM 1.104672 3.822640 0.592966 80.645161


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [ ]:
from google.colab import files
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
import io

# 1. CARGA DE ARCHIVOS
print("Selecciona tus archivos TAMPICO.csv, HOUSTON.csv, MIAMI.csv y MATAMOROS.csv")
uploaded = files.upload()

# --- FUNCIONES DE MÉTRICAS Q1 ---
def calc_smape(y_true, y_pred):
    return 100 * np.mean(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred)))

def calc_da(y_true, y_pred):
    return 100 * np.mean(np.sign(np.diff(y_true)) == np.sign(np.diff(y_pred)))

def calc_mase(y_true, y_pred, y_train_history):
    d = np.abs(np.diff(y_train_history)).sum() / (len(y_train_history) - 1)
    return mean_absolute_error(y_true, y_pred) / d

def create_windows(series, window_size=12):
    X, y = [], []
    for i in range(len(series) - window_size):
        X.append(series[i:i + window_size]); y.append(series[i + window_size])
    return np.array(X), np.array(y)

# 2. PROCESAMIENTO AUTOMÁTICO
res_lgbm = []

for fn in uploaded.keys():
    name = fn.replace('.csv', '').upper()
    print(f"\n>>> Sintonizando LightGBM para {name}...")

    df = pd.read_csv(io.BytesIO(uploaded[fn]))
    series = pd.to_numeric(df.iloc[:, 1], errors='coerce').dropna().values

    # Partición exacta (227/65/32)
    train, val, test = series[0:227], series[227:292], series[292:324]
    history = np.concatenate([train, val])

    window = 12
    X_train, y_train = create_windows(train, window)
    X_val, y_val = create_windows(np.concatenate([train[-window:], val]), window)
    X_hist, y_hist = create_windows(history, window)
    X_test, y_test = create_windows(np.concatenate([history[-window:], test]), window)

    # --- GRID SEARCH POR CIUDAD ---
    best_rmse = float('inf')
    best_p = {}
    for n in [100, 500]:
        for leaves in [7, 15, 31]: # Hojas pequeñas para evitar overfitting
            for lr in [0.01, 0.05]:
                m = lgb.LGBMRegressor(n_estimators=n, num_leaves=leaves, learning_rate=lr, random_state=42, verbosity=-1).fit(X_train, y_train)
                rmse_v = np.sqrt(mean_squared_error(y_val, m.predict(X_val)))
                if rmse_v < best_rmse:
                    best_rmse, best_p = rmse_v, {'n_estimators': n, 'num_leaves': leaves, 'learning_rate': lr}

    # --- EVALUACIÓN FINAL ---
    final_m = lgb.LGBMRegressor(**best_p, random_state=42, verbosity=-1).fit(X_hist, y_hist)
    preds = final_m.predict(X_test)

    # Métricas y Gráfica
    res_lgbm.append({
        "City": name, "Model": "LGBM", "RMSE": np.sqrt(mean_squared_error(test, preds)),
        "sMAPE": calc_smape(test, preds), "MASE": calc_mase(test, preds, history), "DA": calc_da(test, preds)
    })

    plt.figure(figsize=(10, 4))
    plt.plot(test, label='Real', color='black', ls='--')
    plt.plot(preds, label='LGBM Forecast', color='cyan')
    plt.title(f'LightGBM Forecast: {name}'); plt.legend(); plt.grid(True, alpha=0.3); plt.show()

    pd.DataFrame({'Actual': test, 'Forecast': preds}).to_csv(f'{name}_AVG_LGBM_detailed.csv', index=False)

print("\n=== TABLA DE MÉTRICAS LIGHTGBM ===")
print(pd.DataFrame(res_lgbm).to_string(index=False))